In [0]:
# Databricks notebook cell - Python
def f_count_check(database, operation_type, table_name, number_diff):
    # Create a temporary view of the table's history
    catalog = "geeks_az_dbs"
    spark.sql(f"""DESC HISTORY {catalog}.{database}.{table_name}""").createOrReplaceTempView("Table_count")
    
    # Get the output row count for the most recent matching operation
    count_current = spark.sql(f"""
        SELECT operationMetrics.numOutputRows 
        FROM Table_count 
        WHERE version = (
            SELECT MAX(version) 
            FROM Table_count 
            WHERE TRIM(LOWER(operation)) = LOWER('{operation_type}')
        )
    """)
    
    final_count_current = 0
    if count_current.first() is not None and count_current.first().numOutputRows is not None:
        final_count_current = int(count_current.first().numOutputRows)
    
    # Get the previous output row count before the latest operation
    count_previous = spark.sql(f"""
        SELECT operationMetrics.numOutputRows 
        FROM Table_count 
        WHERE version < (
            SELECT version 
            FROM Table_count 
            WHERE TRIM(LOWER(operation)) = LOWER('{operation_type}') 
            ORDER BY version DESC 
            LIMIT 1
        )
        ORDER BY version DESC 
        LIMIT 1
    """)
    
    final_count_previous = 0
    if count_previous.first() is not None and count_previous.first().numOutputRows is not None:
        final_count_previous = int(count_previous.first().numOutputRows)

    records_diff = final_count_current - final_count_previous

    # Check if the difference exceeds the threshold
    if (records_diff) > number_diff:
        pass
       # raise Exception(f"Difference is huge in {table_name}")

    print(f"records diff {records_diff}")

In [0]:

list_table_info = [
    ("STREAMING UPDATE", "plane", 100),
    ("STREAMING UPDATE", "flight", 200),
    ("STREAMING UPDATE", "airport", 100),
    ("STREAMING UPDATE", "cancellation", 100),
    ("STREAMING UPDATE", "unique_carriers", 500),
    ("Write", "airlines", 10),
]
for i in list_table_info:
    f_count_check("cleansed_geekcoders", i[0], i[1], i[2])

In [0]:
%sql
DESC HISTORY geeks_az_dbs.cleansed_geekcoders.plane